# AGAR End-to-End Production Pipeline
Jalankan sel secara berurutan. Edit path konfigurasi sebelum training penuh.

In [ ]:
%cd /kaggle/working/Bacteria-2/agar_production_pipeline
!pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
config = Path('config.yaml')
if not config.exists():
    config.write_text(Path('config.example.yaml').read_text())
print(config.read_text())

## 1. Manifest, outer plate, ROI, dan flat-field sigma 0.04

In [ ]:
!python run_pipeline.py prepare --config config.yaml

## 2. Train dan evaluasi U²-NetP plate localization

In [ ]:
!python run_pipeline.py train-plate --config config.yaml

## 3. Materialisasi tile sekali saja

In [ ]:
!python run_pipeline.py prepare-tiles --config config.yaml

## 4. Train ResNet50-FPN CenterNet-style

In [ ]:
!python run_pipeline.py train-colony --config config.yaml

## 5. Evaluasi detector dan pipeline production mentah


In [ ]:
!python run_pipeline.py evaluate --config config.yaml
!python run_pipeline.py evaluate-e2e --config config.yaml


## 6. Ekspor TorchScript dan Core ML

In [ ]:
!pip install -q -r requirements-coreml.txt
!python run_pipeline.py export --config config.yaml


In [ ]:
import pandas as pd
from pathlib import Path
metrics = Path('../production_artifacts/metrics/all_evaluation_metrics.csv')
display(pd.read_csv(metrics))
print('Output:', metrics.parent.parent.resolve())